### Preamble

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers peft evaluate tomli accelerate -U bitsandbytes

In [ ]:
import os
import re
import json
import time
import random
import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split


import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

from sklearn.metrics import accuracy_score
import evaluate

from peft import LoraConfig, get_peft_model


In [ ]:
torch.set_default_dtype(torch.float32)

In [ ]:
from huggingface_hub import login
HF_ACCESS_TOKEN = "your token here"
login(HF_ACCESS_TOKEN)

### Language

In [ ]:
language = "Spanish" # "English" # "Arabic"

In [ ]:
train = f"/content/drive/MyDrive/CheckThat Task2/{language}/train.jsonl"

if language == "English":
  test = f"/content/drive/MyDrive/CheckThat Task2/{language}/clef2026_gpt4_o_mini_val.json"
elif language == "Spanish":
  test = f"/content/drive/MyDrive/CheckThat Task2/{language}/clef_predictions_1B.json"
elif language == "Arabic":
  test = f"/content/drive/MyDrive/CheckThat Task2/{language}/clef2026_gpt4_o_mini_val_arabic.json"

### Data

In [ ]:
import pandas as pd
import json
import random
import re


unknown_counter = 0
UNKNOWN_LIMIT = 150

def remove_label_pattern(text):
    justification = re.sub(r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))", "", text, flags=re.IGNORECASE).strip()
    return justification.replace("\n", " ")

def sample_training_examples(row):
    global unknown_counter

    label = row["label"].lower()
    verdict_list = [v.lower() for v in row["Verdict_list"]]

    # Step 1: Identify indices of different types
    correct_indices = [i for i, v in enumerate(verdict_list) if v == label]
    true_indices = [i for i, v in enumerate(verdict_list) if v == "true" and v != label]
    false_indices = [i for i, v in enumerate(verdict_list) if v == "false" and v != label]
    conflicting_indices = [i for i, v in enumerate(verdict_list) if v == "conflicting" and v != label]
    unknown_indices = [i for i, v in enumerate(verdict_list) if v == "unknown"]

    selected_indices = []

    # Step 2: Sample correct ones if available
    if len(correct_indices) >= 2:
        selected_indices.extend(random.sample(correct_indices, 2))
        num_remaining = 4
    elif len(correct_indices) == 1:
        selected_indices.append(correct_indices[0])
        num_remaining = 5
    else:
        num_remaining = 6

    # Step 3: Fill remaining slots with diverse wrong answers
    wrong_indices = []
    if label != "true" and true_indices:
        wrong_indices.append(random.choice(true_indices))
    if label != "false" and false_indices:
        wrong_indices.append(random.choice(false_indices))
    if label != "conflicting" and conflicting_indices:
        wrong_indices.append(random.choice(conflicting_indices))

    # Step 4: Ensure diversity while filling remaining slots
    wrong_indices = list(set(wrong_indices))  # Remove duplicates
    needed = num_remaining - len(wrong_indices)

    # Add extra wrong ones if needed
    all_wrong_indices = true_indices + false_indices + conflicting_indices
    random.shuffle(all_wrong_indices)
    wrong_indices.extend(all_wrong_indices[:needed])

    selected_indices.extend(wrong_indices[:num_remaining])

    # Step 5: Handle case where all values are "unknown"
    if not selected_indices and unknown_indices:
        # If everything is unknown, just take 5 unknowns
        selected_indices = random.sample(unknown_indices, min(5, len(unknown_indices)))
    elif unknown_indices and unknown_counter < UNKNOWN_LIMIT:
        # Only pop if there is something to pop
        if selected_indices:
            selected_indices.pop()
        selected_indices.append(random.choice(unknown_indices))
        unknown_counter += 1

    return selected_indices


# data = pd.read_json("reasoning_traces/quantemp_English_train.jsonl", lines=True)
data = pd.DataFrame(pd.read_json(f"/content/drive/MyDrive/CheckThat Task2/{language}/spanish_train.json"))

# Apply function to each row. sampled_indices contain the indices of the sampled dataset
data["sampled_indices"] = data.apply(sample_training_examples, axis=1)


final_training_data = []

for idx in range(len(data)):
    class_label = 0
    item = data.loc[idx]

    label = item['label'].lower()



    for decoding_idx, decoding_sample in enumerate(item["sampled_indices"]):

        justification = remove_label_pattern(item["Reasoning_traces"][decoding_sample])
        verdict = item['Verdict_list'][decoding_sample].lower()

        sample_id = str(idx) + "_" + chr(97 + decoding_idx)


        if verdict == label:
            class_label = 1
        else:
            class_label = 0



        # Handles empty justification
        if len(justification.split(" ")) < 3:
            continue


        final_training_data.append({
            "sample_id": sample_id,
            "input_text": f"Claim: {item['claim']}\nVerdict: {verdict}\nJustification: {justification}",
            "Label": label,
            "Verdict": verdict,
            "Class": class_label})

print(len(final_training_data))
# with open("output/training_data_for_RM/English_train.json", "w") as fp:
#    json.dump(final_training_data, fp, indent = 4)
pd.DataFrame(final_training_data).to_json(
     f"/content/drive/MyDrive/CheckThat Task2/{language}/train.jsonl",
     orient="records",
     lines=True,
     force_ascii=False
 )

print("Finshed preprocessing the data.")

9038
Finshed preprocessing the data.


In [ ]:
import pandas as pd
import json

# Load your raw nested JSON
file_path = f"/content/drive/MyDrive/CheckThat Task2/{language}/clef_2026_checkthat_english_train.json"
with open(file_path, "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)

# 1. Ground Truth Distribution
print("=== Ground Truth Label Distribution ===")
print(df['label'].str.lower().value_counts(normalize=True) * 100)

# 2. Generator Output Distribution
all_generated_verdicts = []
for verdicts in df['Verdict_list']:
    all_generated_verdicts.extend([v.lower() for v in verdicts])

print("\n=== Generated Traces Verdict Distribution ===")
print(pd.Series(all_generated_verdicts).value_counts(normalize=True) * 100)

# 3. The "Trap" Analysis (How often are there ZERO correct traces?)
zero_correct_count = 0
for idx, row in df.iterrows():
    ground_truth = row['label'].lower()
    verdicts = [v.lower() for v in row['Verdict_list']]
    if ground_truth not in verdicts:
        zero_correct_count += 1

print(f"\n=== Impossible Cases ===")
print(f"Claims where the generator produced ZERO correct traces: {zero_correct_count} out of {len(df)}")
print(f"Percentage of dataset permanently trapped: {(zero_correct_count/len(df))*100:.2f}%")

=== Ground Truth Label Distribution ===
label
false          58.078125
conflicting    23.562500
true           18.359375
Name: proportion, dtype: float64

=== Generated Traces Verdict Distribution ===
false          39.850417
conflicting    31.103763
true           29.045820
Name: proportion, dtype: float64

=== Impossible Cases ===
Claims where the generator produced ZERO correct traces: 1917 out of 6400
Percentage of dataset permanently trapped: 29.95%


### Preprocessing

In [ ]:
f1_metric = evaluate.load("f1")

def format_time(elapsed):
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))


def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")


def print_trainable_parameters(model):
    trainable_params, all_param = 0, 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print(
        f"trainable params: {trainable_params} || "
        f"all params: {all_param} || "
        f"trainable%: {100 * trainable_params / all_param:.2f}"
    )


In [ ]:
class CustomClassifier(torch.nn.Module):
    def __init__(
        self,
        model_name,
        num_labels=1,
        hidden_dim=None,
        dropout_value=0.1,
        freeze_base_layer=True,
        use_lora=False,
        is_base_encoder=True,
        lora_rank=8,
        lora_alpha=16,
        quantization_config=None,
        device_map="auto",
        token=None
    ):
        super().__init__()

        self.model = AutoModel.from_pretrained(
            model_name,
            quantization_config=quantization_config,
            device_map=device_map,
            token=token
        )
        self.is_base_encoder = is_base_encoder

        if freeze_base_layer:
            for param in self.model.parameters():
                param.requires_grad = False

        if use_lora:
            lora_config = LoraConfig(
                r=lora_rank,
                lora_alpha=lora_alpha,
                target_modules=["q_proj", "k_proj", "v_proj"],
                lora_dropout=0.0,
                bias="none",
            )
            self.model = get_peft_model(self.model, lora_config)

        hidden_size = self.model.config.hidden_size

        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(hidden_dim, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)

        if self.is_base_encoder:
            pooled_output = outputs.pooler_output
        else:
            pooled_output = outputs.last_hidden_state[:, -1, :]
        pooled_output = pooled_output.to(torch.device("cuda"), dtype=torch.float)
        #self.classifier = self.classifier.to(torch.device("cuda"),torch.float16)
        logits = self.classifier(pooled_output)
        return logits

In [ ]:
class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item


In [ ]:
class TrainerModule:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        tokenizer,
        epochs,
        lr,
        patience,
        output_dir,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.tokenizer = tokenizer
        self.epochs = epochs

        self.optimizer = AdamW(model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch+1}/{self.epochs}")
            self.model.train()

            total_loss, total_acc = 0, 0

            for batch in tqdm(self.train_loader):
                self.optimizer.zero_grad()

                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)
                logits = self.model(input_ids, attention_mask)

                loss = self.loss_fn(logits.squeeze(1), labels)

                loss.backward()
                self.optimizer.step()
                self.scheduler.step()

                total_loss += loss.item()

                preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy()
                total_acc += accuracy_score(labels.cpu().numpy(), preds)

            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Acc: {total_acc / len(self.train_loader):.4f}")

            self.evaluate(epoch)

    def evaluate(self, epoch):
        self.model.eval()
        total_loss, total_acc = 0, 0

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask)
                loss = self.loss_fn(logits.squeeze(1), labels)

                total_loss += loss.item()
                preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy()
                total_acc += accuracy_score(labels.cpu().numpy(), preds)

        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Acc: {total_acc / len(self.val_loader):.4f}")
        self.tokenizer.save_pretrained(self.output_dir)
        torch.save(
            self.model.state_dict(),
            os.path.join(self.output_dir, f"model"),
        )


In [ ]:
class VerifierEvaluator:
    def __init__(
        self,
        model_path,
        tokenizer_path,
        base_model,
        use_decomp,
        device="cuda",
        quantization_config=None,
        token=None
    ):
        super().__init__()
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.use_decomp = use_decomp

        # Pass token to AutoTokenizer.from_pretrained
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, token=token)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = CustomClassifier(
            base_model,
            use_lora=True,
            is_base_encoder=False,
            quantization_config=quantization_config,
            token=token
        )
        self.model.load_state_dict(torch.load(model_path, map_location=self.device), strict=False)
        self.model.to(self.device)
        self.model.eval()

    def encode_input(self, claim, questions, verdict, justification, max_length=150):

        text = f"Claim: {claim}\nVerdict: {verdict}\nJustification: {justification}"

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, questions, verdict, justification):
        ids, mask = self.encode_input(claim, questions, verdict, justification)
        with torch.no_grad():
            return float(self.model(ids, mask).item())

# LoRA Fine-Tuning LLaMA-3.2-Instruct 1B
This notebook fine-tunes a HuggingFace LLaMA-3B-Instruct model using LoRA (PEFT).

### Training

In [ ]:
# ===== PATH =====
TRAIN_CSV = train
VAL_CSV   = test
# TEST_JSON = "/kaggle/input/YOUR_DATA/test.jsonl"

OUTPUT_DIR = "/content/drive/MyDrive/CheckThat Task2"

BASE_MODEL = "meta-llama/Llama-3.2-1B"

MAX_LENGTH = 256
BATCH_SIZE = 4
EPOCHS = 3
LR = 1e-4


In [ ]:
train_df = pd.read_json(TRAIN_CSV, lines=True)

In [ ]:
train_df.head(1)

,sample_id,input_text,Label,Verdict,Class
0,0_a,Claim: توفيت ثلاث نساء جراء الحريق الذي اندلع ...,true,true,1


In [ ]:

train_df = pd.read_json(TRAIN_CSV, lines = True)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["Class"],
    random_state=42
)
# val_df = pd.read_json(VAL_CSV, lines = True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_ACCESS_TOKEN)

train_dataset = TextDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = TextDataset(val_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

model = CustomClassifier(
    BASE_MODEL,
    use_lora=True,
    is_base_encoder=False,
    device_map="auto",
    token=HF_ACCESS_TOKEN
)

print_trainable_parameters(model)

trainer = TrainerModule(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer = tokenizer,
    epochs=EPOCHS,
    lr=LR,
    patience=2,
    output_dir=OUTPUT_DIR,
)

trainer.train()


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

...

Train Loss: 0.3689
Train Acc: 0.8379
Val Loss: 0.3116
Val Acc: 0.8861

Epoch 2/3


100%|██████████| 1808/1808 [02:40<00:00, 11.29it/s]


Train Loss: 0.2011
Train Acc: 0.9199
Val Loss: 0.2165
Val Acc: 0.9148

Epoch 3/3


100%|██████████| 1808/1808 [02:39<00:00, 11.33it/s]


Train Loss: 0.1017
Train Acc: 0.9577
Val Loss: 0.1918
Val Acc: 0.9270


### Predictions

In [ ]:
with open(VAL_CSV, "r") as f:
  test_data = json.load(f)

predictions = []

evaluator = VerifierEvaluator(
    model_path=f"{OUTPUT_DIR}/model",
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    use_decomp=True,
)

# Example scoring
for idx, sample in enumerate(test_data):
  verdict_list = []
  verifier_score_list = []
  justification_list = []
  approved_majority_list = []
  for decoding_sample in range(1, len(sample["Reasoning_traces"]) + 1):
    justification = (
        remove_label_pattern(
            sample["Reasoning_traces"][decoding_sample - 1]
        ).split("Label:")[0]
    )
    score = evaluator.score(
    sample["claim"],
    sample.get("Questions", ""),
    sample["Verdict_list"][decoding_sample - 1].lower(),
    justification,
)
    verdict_list.append(sample["Verdict_list"][decoding_sample - 1])
    justification_list.append(justification)
    verifier_score_list.append(score)
    best_verdict = verdict_list[np.argmax(np.array(verifier_score_list))]
    predictions.append({
        "query_id": idx,
        "Claim": sample["claim"],
        "Label": sample["label"],
        "Verdict_BoN": best_verdict,
        "BoN_Verdict_list": verdict_list,
        "Reasoning_traces": justification_list,
        "score_list": verifier_score_list,
    })




Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
with open(f"/content/drive/MyDrive/CheckThat Task2/{language}/clef_predictions_1B.json", "w") as fp:
  json.dump(predictions, fp, indent=4)

### Visualizing the predictions

In [ ]:
df = pd.read_json(f"/content/drive/MyDrive/CheckThat Task2/{language}/clef_predictions_1B.json")

In [ ]:
df.head(1)

,query_id,Claim,Label,Verdict_BoN,BoN_Verdict_list,Reasoning_traces,score_list
0,0,ناسا ترصد أكثر من 30 نجم في السماء الدنيا سرعت...,False,False,"[False, False, False, False, False, False, Fal...",[الادعاء يقول إن وكالة ناسا ترصد أكثر من 30 نج...,"[2.016681671142578, 3.023765563964843, 4.26496..."


# Fine-Tuning LLaMA-3.2-3B Quantized

In [ ]:
# ===== PATH =====
TRAIN_CSV = train
VAL_CSV   = test
# TEST_JSON = "/kaggle/input/YOUR_DATA/test.jsonl"

OUTPUT_DIR = "/content/drive/MyDrive/CheckThat Task2"

SECOND_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

MAX_LENGTH = 256
BATCH_SIZE = 4
EPOCHS = 3
LR = 1e-4


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer_llm = AutoTokenizer.from_pretrained(SECOND_MODEL, token=HF_ACCESS_TOKEN)


In [ ]:
train_df = pd.read_json(TRAIN_CSV, lines = True)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["Class"],
    random_state=42
)
# val_df = pd.read_json(VAL_CSV, lines = True)

train_dataset = TextDataset(train_df, tokenizer_llm, MAX_LENGTH)
val_dataset = TextDataset(val_df, tokenizer_llm, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

model = CustomClassifier(
    SECOND_MODEL,
    use_lora=True,
    is_base_encoder=False,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_ACCESS_TOKEN
)

print_trainable_parameters(model)

trainer = TrainerModule(
    model,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer = tokenizer_llm,
    epochs=EPOCHS,
    lr=LR,
    patience=2,
    output_dir=OUTPUT_DIR,
)

trainer.train()

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

trainable params: 3214337 || all params: 1806678017 || trainable%: 0.18

Epoch 1/3


100%|██████████| 2021/2021 [10:02<00:00,  3.35it/s]


Train Loss: 0.3531
Train Acc: 0.8122
Val Loss: 0.0945
Val Acc: 0.9708

Epoch 2/3


100%|██████████| 2021/2021 [10:03<00:00,  3.35it/s]


Train Loss: 0.0481
Train Acc: 0.9869
Val Loss: 0.0441
Val Acc: 0.9872

Epoch 3/3


100%|██████████| 2021/2021 [10:02<00:00,  3.36it/s]


Train Loss: 0.0100
Train Acc: 0.9970
Val Loss: 0.0400
Val Acc: 0.9911


In [ ]:
with open(VAL_CSV, "r") as f:
  test_data = json.load(f)

predictions = []

# Ensure bnb_config is defined before this cell or passed correctly if not global
# Assuming bnb_config and HF_ACCESS_TOKEN are available from previous cells
# If not, you might need to re-run the cell where bnb_config is defined.

evaluator = VerifierEvaluator(
    model_path=f"{OUTPUT_DIR}/llama3b/{language}",
    tokenizer_path=SECOND_MODEL,
    base_model=SECOND_MODEL,
    use_decomp=True,
    quantization_config=bnb_config, # Pass the quantization config
    token=HF_ACCESS_TOKEN # Pass the access token
)

# Example scoring
for idx, sample in enumerate(test_data):
  verdict_list = []
  verifier_score_list = []
  justification_list = []
  approved_majority_list = []
  for decoding_sample in range(1, len(sample["Reasoning_traces"]) + 1):
    justification = (
        remove_label_pattern(
            sample["Reasoning_traces"][decoding_sample - 1]
        ).split("Label:")[0]
    )
    score = evaluator.score(
    sample["claim"],
    sample.get("Questions", ""),
    sample["Verdict_list"][decoding_sample - 1].lower(),
    justification,
)
    verdict_list.append(sample["Verdict_list"][decoding_sample - 1])
    justification_list.append(justification)
    verifier_score_list.append(score)
    best_verdict = verdict_list[np.argmax(np.array(verifier_score_list))]
    predictions.append({
        "query_id": idx,
        "Claim": sample["claim"],
        "Label": sample["label"],
        "Verdict_BoN": best_verdict,
        "BoN_Verdict_list": verdict_list,
        "Reasoning_traces": justification_list,
        "score_list": verifier_score_list,
    })


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [ ]:
with open(f"/content/drive/MyDrive/CheckThat Task2/{language}/clef_predictions_3B.json", "w") as fp:
  json.dump(predictions, fp, indent=4)

# Pairwise Ranking

In [ ]:
import random

final_training_data = []

for idx, item in df.iterrows():
    label = item['label'].lower()
    verdicts = [v.lower() for v in item['Verdict_list']]
    traces = [remove_label_pattern(t).split("Label:")[0] for t in item['Reasoning_traces']]

    # Identify indices of correct and incorrect traces
    correct_indices = [i for i, v in enumerate(verdicts) if v == label]
    incorrect_indices = [i for i, v in enumerate(verdicts) if v != label]

    # Skip if we can't form a pair
    if not correct_indices or not incorrect_indices:
        continue

    # Create pairs
    for c_idx in correct_indices:
        # Sample up to 3 incorrect traces to pair against to avoid data explosion
        sampled_incorrect = random.sample(incorrect_indices, min(3, len(incorrect_indices)))

        for inc_idx in sampled_incorrect:
            # 50% chance Correct trace is Candidate A
            if random.choice([True, False]):
                input_text = f"Claim: {item['claim']}\nCandidate A: {traces[c_idx]}\nCandidate B: {traces[inc_idx]}"
                target = 1.0
            # 50% chance Correct trace is Candidate B
            else:
                input_text = f"Claim: {item['claim']}\nCandidate A: {traces[inc_idx]}\nCandidate B: {traces[c_idx]}"
                target = 0.0

            final_training_data.append({
                "sample_id": f"{idx}_{c_idx}_{inc_idx}",
                "input_text": input_text,
                "Class": target
            })

print(f"Total Pairwise Samples Generated: {len(final_training_data)}")

# Save to JSONL for the TrainerModule
pd.DataFrame(final_training_data).to_json(
     f"/content/drive/MyDrive/CheckThat Task2/{language}/train_pairwise.jsonl",
     orient="records",
     lines=True,
     force_ascii=False
)

Total Pairwise Samples Generated: 75690


In [ ]:
class PairwiseVerifierEvaluator:
    def __init__(self, model_path, tokenizer_path, base_model, quantization_config=None, token=None, device="cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, token=token)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        # Load quantized base model
        self.model = CustomClassifier(
            base_model,
            use_lora=True,
            is_base_encoder=False,
            quantization_config=quantization_config,
            token=token,
            # Ensure quantization_config is accepted by CustomClassifier
        )

        # Load weights strictly to LoRA adapters and classifier head
        state_dict = torch.load(model_path, map_location=self.device)
        self.model.load_state_dict(state_dict, strict=False)
        self.model.to(self.device)
        self.model.eval()

    def encode_pair(self, claim, trace_a, trace_b, max_length=512): # Increased max_length for two traces
        text = f"Claim: {claim}\nCandidate A: {trace_a}\nCandidate B: {trace_b}"
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        return encoding["input_ids"].to(self.device), encoding["attention_mask"].to(self.device)

    def score_pair(self, claim, trace_a, trace_b):
        ids, mask = self.encode_pair(claim, trace_a, trace_b)
        with torch.no_grad():
            logit = float(self.model(ids, mask).item())
            # Positive logit = Model prefers A. Negative = Model prefers B.
            return logit

In [ ]:
language = "English"

In [ ]:
# ===== PATH =====
TRAIN_CSV = f"/content/drive/MyDrive/CheckThat Task2/{language}/train_pairwise.jsonl"
VAL_CSV   = f"/content/drive/MyDrive/CheckThat Task2/{language}/clef2026_gpt4_o_mini_val.json"
# TEST_JSON = "/kaggle/input/YOUR_DATA/test.jsonl"

OUTPUT_DIR = "/content/drive/MyDrive/CheckThat Task2"

BASE_MODEL = "meta-llama/Llama-3.2-1B"

MAX_LENGTH = 512
BATCH_SIZE = 4
EPOCHS = 3
LR = 1e-4


In [ ]:
train_df = pd.read_json(TRAIN_CSV, lines = True)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["Class"],
    random_state=42
)
# val_df = pd.read_json(VAL_CSV, lines = True)
tokenizer_llm = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_ACCESS_TOKEN)


train_dataset = TextDataset(train_df, tokenizer_llm, MAX_LENGTH)
val_dataset = TextDataset(val_df, tokenizer_llm, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

model = CustomClassifier(
    BASE_MODEL,
    use_lora=True,
    is_base_encoder=False,
    # quantization_config=bnb_config,
    device_map="auto",
    token=HF_ACCESS_TOKEN
)

print_trainable_parameters(model)

trainer = TrainerModule(
    model,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer = tokenizer_llm,
    epochs=EPOCHS,
    lr=LR,
    patience=2,
    output_dir=OUTPUT_DIR,
)

trainer.train()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

trainable params: 1181697 || all params: 1236996097 || trainable%: 0.10

Epoch 1/3


100%|██████████| 15138/15138 [25:29<00:00,  9.90it/s]


Train Loss: 0.5531
Train Acc: 0.6669
Val Loss: 0.3488
Val Acc: 0.8203

Epoch 2/3


100%|██████████| 15138/15138 [25:31<00:00,  9.89it/s]


Train Loss: 0.2105
Train Acc: 0.9047
Val Loss: 0.1486
Val Acc: 0.9417

Epoch 3/3


100%|██████████| 15138/15138 [25:32<00:00,  9.88it/s]


Train Loss: 0.0696
Train Acc: 0.9754
Val Loss: 0.1025
Val Acc: 0.9672


In [ ]:
from tqdm import tqdm

with open(VAL_CSV, "r") as f:
    test_data = json.load(f)

predictions = []

evaluator = PairwiseVerifierEvaluator(
    model_path=f"{OUTPUT_DIR}/model", # Path to your trained model
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    # quantization_config=bnb_config,
    token=HF_ACCESS_TOKEN
    # pass bnb_config and token if using 3B model
)

for idx, sample in enumerate(tqdm(test_data, desc="Evaluating Pairs")):
    claim = sample["claim"]
    verdicts = sample["Verdict_list"]
    traces = [remove_label_pattern(t).split("Label:")[0] for t in sample["Reasoning_traces"]]

    num_traces = len(traces)
    win_scores = [0.0] * num_traces

    # Round Robin: Compare every trace to every other trace
    for i in range(num_traces):
        for j in range(num_traces):
            if i == j:
                continue

            # Score Trace I vs Trace J
            score = evaluator.score_pair(claim, traces[i], traces[j])

            # If logit is positive, Trace I wins. Add the logit to its score.
            # If logit is negative, Trace J wins. We add absolute logit to J's score.
            if score > 0:
                win_scores[i] += score
            else:
                win_scores[j] += abs(score)

    # The BoN trace is the one with the highest total win score
    best_idx = np.argmax(np.array(win_scores))
    best_verdict = verdicts[best_idx]

    predictions.append({
        "query_id": idx,
        "Claim": claim,
        "Label": sample["label"],
        "Verdict_BoN": best_verdict,
        "BoN_Verdict_list": verdicts,
        "Reasoning_traces": traces,
        "score_list": win_scores, # This feeds perfectly into the organizer's IR metrics script
    })

# Save for the CLEF evaluation script
with open(f"/content/drive/MyDrive/CheckThat Task2/{language}/clef_predictions.json", "w") as fp:
    json.dump(predictions, fp, indent=4)

# Evaluation

Evaluate RM prediction quality on the CLEF dataset.

Metrics computed:
  1. IR-style Recall@k and Precision@k (k=1..5):
       Recall@k    = (# relevant in top-k) / (# relevant in full 15-trace list)
       Precision@k = (# relevant in top-k) / k
     A trace is relevant if its verdict matches the ground truth label.
     Per-sample values saved to per_sample_ir.csv; means saved to result.csv.
  2. Per-class precision, recall, F1 (based on Verdict_BoN vs Label).
  3. Macro and weighted precision, recall, F1.

Output:
  output/RM_prediction/result.csv          — average / aggregate metrics
  output/RM_prediction/per_sample_ir.csv   — per-sample Recall@k and Precision@k

In [ ]:
import csv
import json
import os

from sklearn.metrics import classification_report

INPUT_PATH      = f"/content/drive/MyDrive/CheckThat Task2/{language}/clef_predictions_1B.json"
OUTPUT_PATH     = f"/content/drive/MyDrive/CheckThat Task2/{language}/result.csv"
PER_SAMPLE_PATH = f"/content/drive/MyDrive/CheckThat Task2/{language}/per_sample_ir.csv"
CLASSES         = ["false", "true", "conflicting"]
K_VALUES        = list(range(1, 6))


with open(INPUT_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)

seen: set = set()
unique_data = []
for entry in raw_data:
    if entry["Claim"] not in seen:
        seen.add(entry["Claim"])
        unique_data.append(entry)

print(f"Total unique claims: {len(unique_data)}")


# ── 1. IR-style Recall@k and Precision@k ─────────────────────────────────────
def ir_metrics_at_k(entry: dict, k: int) -> tuple[float, float]:
    """
    Compute IR-style Recall@k and Precision@k for a single claim.

    Recall@k    = (# relevant in top-k) / (# relevant in full 15-trace list)
    Precision@k = (# relevant in top-k) / k

    A trace is relevant when its verdict matches the ground truth label.
    If no trace in the full list is relevant, Recall@k is defined as 0.
    """
    label    = entry["Label"].lower()
    verdicts = [v.lower() for v in entry["BoN_Verdict_list"]]

    ranked = sorted(range(len(entry["score_list"])),
                    key=lambda i: entry["score_list"][i],
                    reverse=True)

    total_relevant     = sum(1 for v in verdicts if v == label)
    top_k_verdicts     = [verdicts[i] for i in ranked[:k]]
    retrieved_relevant = sum(1 for v in top_k_verdicts if v == label)

    recall_k    = retrieved_relevant / total_relevant if total_relevant > 0 else 0.0
    precision_k = retrieved_relevant / k

    return recall_k, precision_k


# Accumulate per-sample records and running sums for means
per_sample_records = []
sum_recall    = {k: 0.0 for k in K_VALUES}
sum_precision = {k: 0.0 for k in K_VALUES}

for entry in unique_data:
    record = {"query_id": entry["query_id"], "claim": entry["Claim"][:80]}
    for k in K_VALUES:
        r, p = ir_metrics_at_k(entry, k)
        record[f"recall@{k}"]    = round(r, 4)
        record[f"precision@{k}"] = round(p, 4)
        sum_recall[k]    += r
        sum_precision[k] += p
    per_sample_records.append(record)

n = len(unique_data)
mean_recall    = {k: sum_recall[k]    / n for k in K_VALUES}
mean_precision = {k: sum_precision[k] / n for k in K_VALUES}


# ── 2 & 3. Classification metrics on Verdict_BoN ─────────────────────────────
y_true = [e["Label"].lower()       for e in unique_data]
y_pred = [e["Verdict_BoN"].lower() for e in unique_data]

report = classification_report(
    y_true, y_pred,
    labels=CLASSES,
    output_dict=True,
    zero_division=0,
)


# ── Save aggregate metrics → result.csv ───────────────────────────────────────
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)

    # Section 1 — IR ranking metrics
    w.writerow(["=== IR Ranking Metrics (RM-ranked reasoning traces) ==="])
    w.writerow(["k", "Mean Recall@k", "Mean Precision@k"])
    for k in K_VALUES:
        w.writerow([k, round(mean_recall[k], 4), round(mean_precision[k], 4)])
    w.writerow([])

    # Section 2 — Per-class classification metrics
    w.writerow(["=== Per-Class Metrics (Verdict_BoN vs Ground Truth) ==="])
    w.writerow(["Class", "Precision", "Recall", "F1-Score", "Support"])
    for cls in CLASSES:
        r = report[cls]
        w.writerow([cls,
                    round(r["precision"], 4),
                    round(r["recall"],    4),
                    round(r["f1-score"],  4),
                    int(r["support"])])
    w.writerow([])

    # Section 3 — Macro & weighted aggregates
    w.writerow(["=== Aggregate Metrics ==="])
    w.writerow(["Average", "Precision", "Recall", "F1-Score"])
    for avg_type in ["macro avg", "weighted avg"]:
        r = report[avg_type]
        w.writerow([avg_type,
                    round(r["precision"], 4),
                    round(r["recall"],    4),
                    round(r["f1-score"],  4)])


# ── Save per-sample IR metrics → per_sample_ir.csv ───────────────────────────
per_sample_header = (
    ["query_id", "claim"]
    + [f"recall@{k}"    for k in K_VALUES]
    + [f"precision@{k}" for k in K_VALUES]
)

with open(PER_SAMPLE_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=per_sample_header)
    w.writeheader()
    w.writerows(per_sample_records)


# ── Print summary ─────────────────────────────────────────────────────────────
print("\n--- IR Ranking Metrics ---")
print(f"  {'k':>3}  {'Mean Recall@k':>14}  {'Mean Precision@k':>16}")
for k in K_VALUES:
    print(f"  {k:>3}  {mean_recall[k]:>14.4f}  {mean_precision[k]:>16.4f}")

print("\n--- Per-Class Metrics (Verdict_BoN) ---")
for cls in CLASSES:
    r = report[cls]
    print(f"  {cls:>12s}:  P={r['precision']:.4f}  R={r['recall']:.4f}  F1={r['f1-score']:.4f}  n={int(r['support'])}")

print("\n--- Aggregate Metrics ---")
for avg_type in ["macro avg", "weighted avg"]:
    r = report[avg_type]
    print(f"  {avg_type}:  P={r['precision']:.4f}  R={r['recall']:.4f}  F1={r['f1-score']:.4f}")

print(f"\nAggregate results  -> {OUTPUT_PATH}")
print(f"Per-sample results -> {PER_SAMPLE_PATH}")


Total unique claims: 562

--- IR Ranking Metrics ---
    k   Mean Recall@k  Mean Precision@k
    1          0.0488            0.6886
    2          0.1005            0.6895
    3          0.1475            0.6851
    4          0.1913            0.6819
    5          0.2401            0.6794

--- Per-Class Metrics (Verdict_BoN) ---
         false:  P=0.8838  R=0.6913  F1=0.7758  n=473
          true:  P=0.0920  R=0.2105  F1=0.1280  n=38
   conflicting:  P=0.1429  R=0.2941  F1=0.1923  n=51

--- Aggregate Metrics ---
  macro avg:  P=0.3729  R=0.3987  F1=0.3654
  weighted avg:  P=0.7630  R=0.6228  F1=0.6790

Aggregate results  -> /content/drive/MyDrive/CheckThat Task2/Spanish/result.csv
Per-sample results -> /content/drive/MyDrive/CheckThat Task2/Spanish/per_sample_ir.csv


In [ ]:
!pip install -q trl peft transformers